# 05 — Bangun Dataset Modeling

Mengubah hasil anotasi Label Studio menjadi split **train/val/test** yang dipakai
identik oleh SVM dan IndoBERT.

**Prasyarat:** ekspor anotasi dari Label Studio (UI: *project > Export > JSON*) dan
simpan ke `outputs/labeling/label_studio_export.json`.

Pipeline sel di bawah:
1. Parse ekspor LS → DataFrame berlabel + cek keseimbangan kelas.
2. Preprocessing dua jalur (`text_svm`, `text_bert`) + split stratified 70/20/10.
3. Simpan ke `data/processed/splits/`.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from configs.config import Config
from src.modeling.labels import parse_label_studio_export, class_distribution
from src.modeling.dataset import build_and_save, build_modeling_frame


In [ ]:
# 1) Parse ekspor Label Studio
df = parse_label_studio_export(Config.label_studio.EXPORT_FILE)
print(f"Total komentar berlabel: {len(df)}")
print("\nDistribusi kelas:")
print(class_distribution(df).to_string())


### Cek keseimbangan

Target dataset **seimbang 1.000/kelas**. Jika satu kelas masih jauh di bawah target,
lanjutkan labeling dulu sebelum melatih (terutama untuk kelas minoritas).

In [ ]:
# 2) Preprocessing dua jalur + split stratified + simpan
df_train, df_val, df_test = build_and_save(df)

print("\nDistribusi per split (harus proporsional):")
for name, part in [("train", df_train), ("val", df_val), ("test", df_test)]:
    print(f"\n[{name}]  n={len(part)}")
    print(part["label"].value_counts().to_string())


In [ ]:
# Intip contoh hasil preprocessing dua jalur
import pandas as pd
pd.set_option("display.max_colwidth", 70)
df_train[["text", "text_svm", "text_bert", "label"]].head(8)


### Trace transformasi langkah-demi-langkah (SVM vs IndoBERT)

Lihat bagaimana satu komentar berubah di tiap tahap. **SVM** lewat 4 tahap
(clean agresif → normalisasi slang → buang stopword → stemming Sastrawi);
**IndoBERT** cukup 1 tahap (clean minimal, morfologi & tanda baca dipertahankan).
Ganti `CONTOH` dengan komentar apa pun untuk mencoba sendiri.

In [ ]:
from src.text_normalizer import (
    clean_for_svm, clean_for_bert, normalize_slang, tokenize, remove_stopwords,
)
from src.modeling.dataset import _get_stemmer

_stemmer = _get_stemmer()


def trace_preprocess(text: str) -> None:
    """Cetak transformasi tiap tahap untuk kedua jalur."""
    print(f"RAW             : {text!r}\n")

    print("── JALUR SVM (agresif, untuk TF-IDF) ──")
    s1 = clean_for_svm(text)
    s2 = normalize_slang(s1)
    s3 = " ".join(remove_stopwords(tokenize(s2)))
    s4 = _stemmer.stem(s3) if _stemmer else s3
    print(f"  1. clean_for_svm  : {s1!r}")
    print(f"  2. normalize_slang: {s2!r}")
    print(f"  3. remove_stopword: {s3!r}")
    print(f"  4. stem (FINAL)   : {s4!r}\n")

    print("── JALUR IndoBERT (minimal, morfologi utuh) ──")
    print(f"  1. clean_for_bert (FINAL): {clean_for_bert(text)!r}")


CONTOH = "Persidangan belum dilaksanakan kok sdh ngajuin ahli, kalo gini rakyat dibikin binggung"
trace_preprocess(CONTOH)


Split tersimpan di `data/processed/splits/{train,val,test}.parquet`.

**Lanjut:**
- `06_train_svm.ipynb` — latih SVM (lokal).
- `07_indobert_finetune_colab.ipynb` — fine-tune IndoBERT (Colab/Kaggle, GPU). Unggah
  ketiga file parquet tadi ke Colab.